# 性能分析

调优以**采集数据为依据**：先看耗时与流水指标确定瓶颈，再选优化手段；每次优化以功能精度正确为前提。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 工具与适用场景

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">工具</th><th style="text-align:left;">适用场景</th><th style="text-align:left;">产物</th></tr>
<tr><td><code>torch_npu.profiler</code></td><td>Python 代码内控制采集范围</td><td>Kernel 耗时、AI Core 流水指标、任务时间线</td></tr>
<tr><td>msOpProf（<code>msprof op</code>）</td><td>单算子上板深度分析</td><td>算子耗时、AI Core 流水、内存/Cache、资源冲突、核内流水图</td></tr>
</table>


## 2. 建立性能基线（可比较的前提）

1. 固定输入 Shape、dtype、TilingKey、block_dim、Stream 与同步位置；
2. **JIT 预热**：先跑一次完成编译，再采集（否则你测的是编译时间）；
3. 测纯 Kernel 耗时——输入分配、随机数、H2D 拷贝、结果校验**不计入**；
4. 多次测量排除波动；优化前后采集配置相同；记录时注明测量口径。


## 3. torch_npu.profiler 采集

```python
import torch_npu

experimental_config = torch_npu.profiler._ExperimentalConfig(
    export_type=[torch_npu.profiler.ExportType.Text],
    profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
    aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
)

kernel[None, block_dim](*args)          # JIT 预热
torch.npu.synchronize()

with torch_npu.profiler.profile(
    activities=[torch_npu.profiler.ProfilerActivity.NPU],
    schedule=torch_npu.profiler.schedule(wait=0, warmup=1, active=5, repeat=1),
    experimental_config=experimental_config,
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(
        "./profiling_output", analyse_flag=True),
) as prof:
    for _ in range(6):  # 1 次预热，5 次采集
        kernel[None, block_dim](*args)
        prof.step()
    torch.npu.synchronize()
```

结果文件检查：`kernel_details.csv` 看 Duration/Block Dim/AI Core 流水指标（定位目标 Kernel 行）；`trace_view.json` 看应用/CANN/NPU 任务时间关系。


## 4. msOpProf 采集

```bash
msprof op --kernel-name='*target_kernel*' --aic-metrics=PipeUtilization python3 test_example.py
```

把 `target_kernel` 换成目标 Kernel 名称；脚本内先完成 JIT 预热。若不指定 `--kernel-name`，可能采到输入构造阶段的 PyTorch Kernel。多组指标采集保持输入与运行条件不变。核内流水采集命令参考 msOpProf 用户指南。


## 5. 理论耗时估算：给"四问"一把尺子

判断"流水占用低不低"，先算**理论值**（Ascend C 性能分析同款方法）：

```text
搬运类流水（MTE1/MTE2/MTE3）理论耗时 ≈ 数据量 ÷ 对应通路带宽
计算类流水（Cube/Vector）理论耗时 ≈ 计算量(FLOPs/OPs) ÷ 峰值算力
```

- 数据量按**实际搬运字节数**累加（含分形对齐扩大的边界，第 4 章）；
- 理论下界 ÷ 实测耗时可作粗略效率指标（同一计量范围）：接近 1 表示接近该模型下界；远小于 1 表示尚有开销或模型未覆盖的因素，不能单凭它判定空等。实测若小于所算“下界”，应先检查字节数、缓存、带宽和测量口径；
- 把四问的每个答案都配上同口径的效率估计与时间线证据，瓶颈判断从感觉变成算术。


## 6. 分析视角

拿到数据后按顺序问四个问题：

1. **耗时**：Kernel 总时长与任务最多核的时长是否一致（负载均衡）；
2. **流水**：AI Core 各 Pipe（MTE2/MTE1/M/FIX、V）的占用率——哪条流水空等；
3. **访存**：带宽利用率、Cache 命中——搬运是否成为瓶颈；
4. **任务线**：时间线上各核任务的起止错位——多核空窗。

四个问题分别指向四类优化：切分重排、流水重叠（下一节）、搬运/布局优化、调度对齐。

访存视角的四个补充检查点（概念级，950 实测确认）：

1. **对齐**：Tile 地址与数据起址满足各 Buffer 对齐要求（UB/L1 32B、L0A/B 512B、L0C 64B，第 2 章）；其中强制对齐是正确性约束，违反时可能被拒绝或产生非法访问，不能只看性能；
2. **UB Bank 冲突**：取决于具体 bank 映射、访问宽度和同时访问的地址，不能用“stride 是 bank 宽度倍数”单独判定（第 3 章 Reg 计算的动机之一）——调整 Tile 形状或 layout 可规避；
3. **多核同地址竞争**：多核写同一 GM 区域（计数器、共享状态）会串行化——尽量让各核写不相交区域，共享更新用原子并减少频次；
4. **数据缓存一致性**：`pl.system.dcci` 清理并失效数据缓存，不是通用的 L2 性能调优开关，也不替代流水同步/跨核事件。仅在通信协议需要时按 API 使用，频繁全缓存操作可能降低性能。


### 可复现的三版本实验

本章 [benchmark_fusion.py](./src/benchmark_fusion.py) 使用第 5 章相同的输入与 tanh GELU Golden，比较单槽、手动双槽和自动流水。每个版本先编译执行并校验，再预热 5 次，记录 5 组、每组 20 次调用的平均墙钟耗时及其中位数；输入构造和 Golden 不计入窗口。

从章节目录执行 `python src/benchmark_fusion.py`，实际结果保存为 `benchmark_ch08.json`；加 `--profile` 时单独采集 `profile_ch08/`。墙钟口径含 Host 下发与同步，不能用它声称纯设备时间；分析设备热点应看 profiler 产物中的具体 Kernel 记录。开启 profiling 的时间不混入普通基线。

比较时同时看最大误差、workspace 字节数和时间分布。如果仅更快一次、其余重叠明显，应增加独立重复或控制环境，而不是宣布稳定收益。

自动版使用 UB 交接，另外两个版本使用 GM workspace；JSON 的 intermediate_storage 标明这一差异。三者输入输出规格一致，但耗时差异包含数据通路变化，不能单独解释为自动调度收益。


## 7. 课后练习

1. （填空）采集前必须先 ______，否则测到的是 ______。
2. （多选）性能基线需要固定哪些条件？
   A. Shape/dtype/TilingKey/block_dim<br/>B. Stream 与同步位置<br/>C. 输入数值的 seed
3. （实践）为你的 GEMM 设计一份"性能记录表"：固定 M/N/K 与 block_dim，记录 Kernel 耗时、Cube 流水占用、Vector 流水占用三列。

In [ ]:
!cat ./answer/08.04_answer.txt

## 8. 本节小结

- 两工具：profiler（代码内、快）/ msOpProf（深度、核内流水）；
- 基线四纪律：固定条件、**JIT 预热**、纯 Kernel 口径、多次测量；
- 分析四问：耗时均衡、流水占用、访存带宽、任务错位——对应四类优化。